# Extra direction features

Pure feature definitions: no fitting, label access or file writes.

Use centred half-windows of 3 and 5 seconds (up to 6 and 10 seconds total). Wider-window displacement averages motion over more GPS observations than the original short heading. Reliability is net displacement divided by travelled GPS-polyline distance. Turn context compares incoming and outgoing directions in the 3-second half-window.

Signed progress projects the window endpoints onto the directed candidate segment, clips at its endpoints, then divides their signed difference by elapsed time. Forward is positive, backward negative. It is not a routed speed estimate.

Require observations before and after the current point. Undefined headings, trip boundaries and zero-length segments remain missing. A stationary valid window has speed/progress zero and reliability zero.

In [ ]:
import numpy as np
import pandas as pd

direction_columns = [
    "heading_diff_smoothed_3s", "signed_progress_mps_3s", "direction_reliability_3s",
    "smoothed_speed_mps_3s", "turn_angle_3s",
    "heading_diff_smoothed_5s", "signed_progress_mps_5s", "direction_reliability_5s",
]


def direction_difference(a, b):
    return np.abs((a - b + 180) % 360 - 180)


def direction_features(candidates, nodes_xy, arc_nodes, gps_xy, time):
    time = np.asarray(time, dtype=float) - time[0]
    assert (np.diff(time) >= 0).all()
    p = candidates["point_id"].to_numpy(dtype=int)
    s = candidates["segment_id"].to_numpy(dtype=int)
    start = nodes_xy[arc_nodes[s, 0]]
    vector = nodes_xy[arc_nodes[s, 1]] - start
    length = np.linalg.norm(vector, axis=1)
    unit = np.divide(vector, length[:, None], out=np.full_like(vector, np.nan), where=length[:, None] > 0)
    road_heading = np.degrees(np.arctan2(vector[:, 0], vector[:, 1])) % 360
    road_heading[length == 0] = np.nan
    path = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(gps_xy, axis=0), axis=1))]
    index = np.arange(len(time))
    features = pd.DataFrame(index=candidates.index)

    for seconds in [3, 5]:
        left = np.searchsorted(time, time - seconds, side="left")
        right = np.searchsorted(time, time + seconds, side="right") - 1
        elapsed = time[right] - time[left]
        span_valid = (left < index) & (right > index) & (elapsed > 0)
        displacement = gps_xy[right] - gps_xy[left]
        distance = np.linalg.norm(displacement, axis=1)
        speed = np.divide(distance, elapsed, out=np.full(len(time), np.nan), where=elapsed > 0)
        speed[~span_valid] = np.nan
        heading = np.degrees(np.arctan2(displacement[:, 0], displacement[:, 1])) % 360
        heading[~span_valid | (distance < 1) | (speed < 0.5)] = np.nan
        travelled = path[right] - path[left]
        reliability = np.divide(distance, travelled, out=np.zeros(len(time)), where=travelled > 0)
        reliability = np.clip(reliability, 0, 1)
        reliability[~span_valid] = np.nan

        along_left = np.clip(np.sum((gps_xy[left[p]] - start) * unit, axis=1), 0, length)
        along_right = np.clip(np.sum((gps_xy[right[p]] - start) * unit, axis=1), 0, length)
        progress = np.divide(along_right - along_left, elapsed[p],
            out=np.full(len(p), np.nan), where=span_valid[p] & (length > 0))
        features[f"heading_diff_smoothed_{seconds}s"] = direction_difference(heading[p], road_heading)
        features[f"signed_progress_mps_{seconds}s"] = progress
        features[f"direction_reliability_{seconds}s"] = reliability[p]
        if seconds == 3:
            features["smoothed_speed_mps_3s"] = speed[p]
            incoming = gps_xy - gps_xy[left]
            outgoing = gps_xy[right] - gps_xy
            a = np.degrees(np.arctan2(incoming[:, 0], incoming[:, 1]))
            b = np.degrees(np.arctan2(outgoing[:, 0], outgoing[:, 1]))
            turn = direction_difference(a, b)
            valid_turn = span_valid & (np.linalg.norm(incoming, axis=1) >= 1) & (np.linalg.norm(outgoing, axis=1) >= 1)
            turn[~valid_turn] = np.nan
            features["turn_angle_3s"] = turn[p]
    return features[direction_columns].astype("float32")